In [ ]:
import pandas as pd
from metloom.pointdata import SnotelPointData
from metloom.variables import SnotelVariables
import numpy as np
import datetime as dt

In [ ]:
dt.datetime.today().weekday() != 3

In [2]:
fx_date = pd.Timestamp('2025-12-04 00:00:00', tz='UTC') + pd.Timedelta(days=1)
end_date = fx_date + pd.Timedelta(days=3)
sites ={
    "Mt. Baker Ski Area" : "909:WA:SNTL",
    "Stevens Pass" : "791:WA:SNTL",
    "Blewett Pass" : "352:WA:SNTL",
    "Snoqualmie Pass" : "672:WA:SNTL",
    "Crystal Mountain" : "418:WA:SNTL",
    "Paradise (Mt. Rainier)" : "679:WA:SNTL",
    "White Pass" : "863:WA:SNTL",
    "Hurricane Ridge" : "974:WA:SNTL"
}

In [3]:

snotel_point = SnotelPointData("515:WA:SNTL","site_name")
snotel_point_2 = SnotelPointData("998:WA:SNTL","Easy Pass")
sntl_df = snotel_point.get_daily_data(fx_date, end_date, [snotel_point.ALLOWED_VARIABLES.SNOWDEPTH,
                                                                 snotel_point.ALLOWED_VARIABLES.SWE,
                                                                 snotel_point.ALLOWED_VARIABLES.TEMPMAX,
                                                                 snotel_point.ALLOWED_VARIABLES.TEMPMIN,
                                                                 snotel_point.ALLOWED_VARIABLES.TEMPAVG,
                                                                 snotel_point.ALLOWED_VARIABLES.PRECIPITATION])


                                                        

In [4]:
sntl_df.columns

Index(['geometry', 'SNOWDEPTH', 'SNOWDEPTH_units', 'SWE', 'SWE_units',
       'MAX AIR TEMP', 'MAX AIR TEMP_units', 'MIN AIR TEMP',
       'MIN AIR TEMP_units', 'AVG AIR TEMP', 'AVG AIR TEMP_units',
       'PRECIPITATION', 'PRECIPITATION_units', 'datasource'],
      dtype='object')

In [9]:
def new_snow_estimate(swe, temp):
    """Estimate snow density based on SWE and temperature using a simple empirical formula."""
    # convert temp to Kelvin
    temp = ((temp - 32) * 5.0/9.0) + 273.15
    # convert swe to m
    swe = swe * 25.4
    # Simple empirical formula for snow density estimation
    if temp <= 258.16:
        density = .50  # Very light, fluffy snow
    elif temp > 273.16:
        density = .250  # Wet, heavy snow
    else:
        density = 0.05 + 0.0017*((temp-258.16)**1.5) 
    print(density)
    new_snow_depth = ((swe) / density) 
    return round(new_snow_depth/25.4, 1)

In [11]:
sntl_df = sntl_df[sntl_df['SWE'] >= 0]
# remove any really big numbers (> 200 inches)
sntl_df = sntl_df[sntl_df['SWE'] <= 200]
swe_change = sntl_df['SWE'].diff().clip(lower=0).sum()
print(swe_change)
# estimate new snow depth from swe and temperature
if "MAX AIR TEMP" and "MIN AIR TEMP" not in sntl_df.columns:
    print("Temperature data not available for snow depth estimation.")
    max_snow_depth_estimate_swe = swe_change / 0.08 # assume 10% density
    min_snow_depth_estimate_swe = swe_change / 0.25 # assume 25% density
    snowdepth_change_swe = swe_change/0.15

    print("assumed snow depth change:", snowdepth_change_swe)
else:        
    if "PRECIPITATION" in sntl_df.columns:
        # lapse assumption:
        lapse = 1 # feet
        feet_to_meters = 3.28084
        lapse_rate_per_C = 6
        precip_total = sntl_df['PRECIPITATION'].diff().clip(lower=0).sum()
        # assume lapse rate for temperature
        lapse_rate_F_1000ft = ((lapse_rate_per_C*9/5)) / feet_to_meters * lapse # 6.8C per 1000m in F per 1000ft
        lapse_adjusted_temp = sntl_df['AVG AIR TEMP'] - lapse_rate_F_1000ft
        if (lapse_adjusted_temp.mean() < 33) and (sntl_df['AVG AIR TEMP'].mean() > 35):
            swe_change = precip_total
            print("using precip based swe change:", swe_change)
        
        max_snow_depth_estimate_swe = new_snow_estimate(swe_change, sntl_df['MIN AIR TEMP'].mean())
        min_snow_depth_estimate_swe = new_snow_estimate(swe_change, sntl_df['MAX AIR TEMP'].mean())
        snowdepth_change_swe = np.mean([max_snow_depth_estimate_swe, min_snow_depth_estimate_swe])
        print("snowdepth change from swe and temp:", snowdepth_change_swe)
        

if ("SNOWDEPTH" in sntl_df.columns) and not (sntl_df['SNOWDEPTH'].isnull().all()):
    sntl_df = sntl_df[sntl_df['SNOWDEPTH'] >= 0]
    sntl_df = sntl_df[sntl_df['SNOWDEPTH'] <= 1000]
    max_snow_depth_estimate = sntl_df['SNOWDEPTH'].diff().clip(lower=0).sum()
    # settlement rate
    density = swe_change / max_snow_depth_estimate
    # density change  per 
    if density < 0.25:
        density += 0.01 * 24 * 4 / 2.54
        # max density is 0.25
        if density > 0.25:
            density = 0.25
    min_snow_depth_estimate = swe_change / density
    snowdepth_change = np.mean([min_snow_depth_estimate, max_snow_depth_estimate])

    if snowdepth_change < snowdepth_change_swe*0.8:
        snowdepth_change = snowdepth_change_swe
    elif snowdepth_change > snowdepth_change_swe*2:
        snowdepth_change = snowdepth_change_swe
    if min_snow_depth_estimate < min_snow_depth_estimate_swe*0.8:
        min_snow_depth_estimate = min_snow_depth_estimate_swe
    elif min_snow_depth_estimate > min_snow_depth_estimate_swe*2:
        min_snow_depth_estimate = min_snow_depth_estimate_swe
    if max_snow_depth_estimate < max_snow_depth_estimate_swe*0.8:
        max_snow_depth_estimate = max_snow_depth_estimate_swe
    elif max_snow_depth_estimate > max_snow_depth_estimate_swe*2:
        max_snow_depth_estimate = max_snow_depth_estimate_swe
    print("snowdepth change from snowdepth data:", snowdepth_change)
else:
    snowdepth_change = snowdepth_change_swe
    min_snow_depth_estimate = min_snow_depth_estimate_swe
    max_snow_depth_estimate = max_snow_depth_estimate_swe
    print("no snowdepth data, using swe based estimate:", snowdepth_change)

# replace any nans with 0
snowdepth_change = 0 if pd.isna(snowdepth_change) else snowdepth_change
min_snow_depth_estimate = 0 if pd.isna(min_snow_depth_estimate) else min_snow_depth_estimate
max_snow_depth_estimate = 0 if pd.isna(max_snow_depth_estimate) else max_snow_depth_estimate

snowdepth_change = round(float(snowdepth_change), 1)
min_snow_depth_estimate = round(float(min_snow_depth_estimate), 1)
max_snow_depth_estimate = round(float(max_snow_depth_estimate), 1)
print("Final snow depth change estimate:", snowdepth_change)

3.3999999999999986
0.10920933877030053
0.13871771168482203
snowdepth change from swe and temp: 27.8
snowdepth change from snowdepth data: 27.8
Final snow depth change estimate: 27.8


In [72]:
max_snow_depth_estimate

26.9